# Genetic drift in mitochondrial segregation

In [ ]:
try:
    import au_molecular_genetics
    print("Already installed")
except ImportError:
    %pip install -q "au_molecular_genetics @ git+https://github.com/au-mbg/molecular_genetics.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1 Exploring variation from mother to oocytes.

Placeholder for figure.


Even though the mother has a fixed mtDNA composition, random sampling
during oocyte formation leads to substantial variation between oocytes,
the simulation below explores this.

The simulation has two parameters:

-   `maternal_heteroplasmy`: This is the fraction of disease-causing
    mtDNA in the mother’s cells.
-   `n_samples`: The number of oocytes sampled, the default of `10000`
    corresponds to more oocytes than a single woman produces whereas a
    number like 500 could be considered a snapshot from a single woman.

You may notice that the `sample_oocyte`-function can take an additional
parameter `N`, `N` is an effective bottleneck size: the number of mtDNA
genomes that effectively found the oocyte population - in humans that
number is roughly 30.

In [ ]:
def sample_oocyte(maternal_heteroplasmy, n_samples=1, N=30):
    oocyte_sample = np.random.binomial(n=N, p=maternal_heteroplasmy, size=n_samples)
    p_oocyte = oocyte_sample / N
    return oocyte_sample, p_oocyte


## Parameters
maternal_heteroplasmy = 0.3 # Maternal heteroplasmy.
n_samples = 10000 # Number of oocytes sampled.

## Sampling
sample, p_ooc = sample_oocyte(maternal_heteroplasmy, n_samples)

## Plot 
fig, ax = plt.subplots()
N = 30
bins = np.linspace(-0.5/N, 1 + 0.5/N, N + 2)
ax.hist(p_ooc, bins=bins, edgecolor='black')
ax.set_xlabel('Oocyte mtDNA disease fraction')
ax.set_ylabel('Count')
plt.show()

#### Exercise 1

The number of oocytes produced by a woman during her lifespan is roughly
500. Run the simulation multiple times with `n_samples = 500` and
consider the variation in oocyte outcomes for mothers with the same
`maternal_heteroplasmy`. Does the *true* distribution change, or just
our estimate of it?

If women produced fewer oocytes, say 100, what changes about what you
observe from one woman if she produces 100 oocytes instead of 500?

#### Exercise 2

Vary `maternal_heteroplasmy` through `0.1`, `0.3` and `0.6`. How does
the center of the distribution change? Does the width of the
distribution change significantly?

#### Exercise 3

Two mothers have the same average mtDNA disease fraction. One produces
children with very different outcomes, the other does not. Why can that
happen?

## 2 Oocyte disease model

A simple model of the risk of disease development in a child, based on
the disease allele frequency in the oocyte is a threshold model. That is

-   **Case 1:** Low disease allele frequency ($f < t_1$) $\rightarrow$
    Healthy child
-   **Case 2:** Intermediate disease allele frequency
    ($t_1 \leq f < t_2$) $\rightarrow$ Diseased child
-   **Case 3:** High allele frequency in the oocyte ($f \geq t_2$)
    $\rightarrow$ Severely affected child

The cell below implements such a model, where you can control the
parameters of oocyte sampling and the threshold model.

In [ ]:
def threshold_model(p, t1, t2):
    state = np.zeros_like(p)        # Case 1
    state[(p >= t1) & (p < t2)] = 1 # Case 2
    state[p >= t2] = 2              # Case 3
    return state

def model_plot(p_ooc, child_state, t1, t2, bins, n_samples):

    states = {0: 'Healthy', 1: 'Diseased', 2: 'Severely affected'}

    ## Plotting
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    ## Left plot of oocyte distributions
    ax = axes[0]
    ax.hist(p_ooc, bins=bins, edgecolor='black') # Plots the distribution

    ## Adds lines and shading
    ax.axvline(t1, color='C1', zorder=-1)
    ax.axvline(t2, color='C2', zorder=-1)
    ax.axvspan(0, t1, color='C0', alpha=0.4, zorder=-1)
    ax.axvspan(t1, t2, color='C1', alpha=0.4, zorder=-1)
    ax.axvspan(t2, 1, color='C2', alpha=0.4, zorder=-1)

    # Annotation arrows.
    ax.annotate('Healthy', (t1/2, 0.95), (0.7, 0.95), xycoords=(ax.transAxes, ax.transAxes), arrowprops=dict(arrowstyle='->', color='C0', lw=1.5), va='center', ha='center', color='C0', fontsize=12)
    ax.annotate('Diseased', ((t1+t2)/2, 0.85), (0.7, 0.85), xycoords=(ax.transAxes, ax.transAxes), arrowprops=dict(arrowstyle='->', color='C1', lw=1.5), va='center', ha='center', color='C1', fontsize=12)
    ax.annotate('Severely\naffected', (t2+0.1, 0.75), (t2+0.3, 0.75), xycoords=(ax.transAxes, ax.transAxes), arrowprops=dict(arrowstyle='->', color='C2', lw=1.5), va='center', ha='center', color='C2', fontsize=12)

    ## Labels and limits
    ax.set_xlabel('Oocyte mtDNA disease fraction')
    ax.set_ylabel('Count')
    ax.set_xlim([0, 1])

    ## Right plot of child outcomes.
    ax = axes[1]
    for state, desc in states.items():
        # Makes a bar for each category by counting the number in each state.
        ax.bar(state, np.sum(child_state == state)/n_samples, width=0.8, edgecolor='black')

    ax.set_xticks(list(states.keys()))
    ax.set_xticklabels(list(states.values()))
    ax.set_ylabel('Probability')
    ax.set_title('Child outcome distribution')
    ax.set_ylim([0, 1])
    plt.show()

The next cell produces a plot using the model with the given parameters

In [ ]:
## Parameters
maternal_heteroplasmy = 0.2
n_samples = 10000

t1 = 0.3 # Disease threshold
t2 = 0.4 # Severe threshold

## Run the simulation
sample, p_ooc = sample_oocyte(maternal_heteroplasmy, n_samples)
child_state = threshold_model(p_ooc, t1, t2)
model_plot(p_ooc, child_state, t1, t2, bins, n_samples)

#### Exercise 4

With `maternal_heteroplasmy = 0.2`, `t1 = 0.3` and `t2 = 0.4`, the
mother’s mtDNA disease fraction (`maternal_heteroplasmy`) is below the
disease threshold. Why do diseased or severely affected children still
appear in the simulation?

#### Exercise 5

Based on the simulations can you explain how a disease might skip a
generation? That is for example a mother that has disease free children
but some of her grandchildren are diseased.